# Credit Card Approval Prediction - Epic 3: Data Preprocessing & Feature Engineering

This notebook implements the complete data preprocessing, cleaning, feature engineering, and normalization pipeline as required by Phase 4 and Phase 5 of the evaluation metrics.

## Preprocessing Actions:
1. **Drop Duplicate Records**: Clean duplicate logs in both demographic and credit datasets.
2. **Handle Missing Values**: Impute missing features (like Occupation Type).
3. **Aggregate Credit Logs**: Group history logs by applicant ID and construct the target label `y` based on delinquency.
4. **Feature Engineering**:
   - `Age_Years`: Convert DAYS_BIRTH to positive years.
   - `Employed_Years`: Convert negative DAYS_EMPLOYED to positive years (unemployed set to 0.0).
   - `Family_Dependency`: Ratio of children to total family members.
   - `Income_Category`: Binned Income levels ('Low', 'Medium', 'High', 'Very High') based on quantiles.
   - `Credit_Window`: Length of credit history in months (credit aggregate).
   - `Open_Month`: Oldest month balance record (credit aggregate).
   - `End_Month`: Latest month balance record (credit aggregate).
   - `Risk_Score`: Average status-mapped risk score (credit aggregate).
5. **Data Merging**: Join demographics and credit history aggregates.
6. **Prevent Data Leakage**: Exclude credit-history aggregates (`Credit_Window`, `Open_Month`, `End_Month`, `Risk_Score`) from training features `X` (as new applicants do not have bank credit card logs yet), but keep demographic features including engineered ones.
7. **Categorical Handling**: Label encode categoricals and save encoders.
8. **Normalization**: Fit and save standard scaler, export `processed_dataset.csv`.

## 1. Import Core Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load Datasets and Clean Duplicates

In [2]:
app_df = pd.read_csv(os.path.join('..', '03_Dataset', 'application_record.csv'))
credit_df = pd.read_csv(os.path.join('..', '03_Dataset', 'credit_record.csv'))

print(f"Application rows before dropping duplicates: {app_df.shape[0]}")
app_df.drop_duplicates(inplace=True)
print(f"Application rows after dropping duplicates: {app_df.shape[0]}")

print(f"Credit history rows before dropping duplicates: {credit_df.shape[0]}")
credit_df.drop_duplicates(inplace=True)
print(f"Credit history rows after dropping duplicates: {credit_df.shape[0]}")

Application rows before dropping duplicates: 5000
Application rows after dropping duplicates: 5000
Credit history rows before dropping duplicates: 177982
Credit history rows after dropping duplicates: 177982


## 3. Impute Missing Values

In [3]:
print("Missing values count in Application:")
print(app_df.isnull().sum())

# Fill Occupation Type missing values with 'Unknown'
app_df['OCCUPATION_TYPE'] = app_df['OCCUPATION_TYPE'].fillna('Unknown')
print(f"Remaining missing values in occupation: {app_df['OCCUPATION_TYPE'].isnull().sum()}")

Missing values count in Application:
ID                        0
CODE_GENDER               0
FLAG_OWN_CAR              0
FLAG_OWN_REALTY           0
CNT_CHILDREN              0
AMT_INCOME_TOTAL          0
NAME_INCOME_TYPE          0
NAME_EDUCATION_TYPE       0
NAME_FAMILY_STATUS        0
NAME_HOUSING_TYPE         0
DAYS_BIRTH                0
DAYS_EMPLOYED             0
FLAG_MOBIL                0
FLAG_WORK_PHONE           0
FLAG_PHONE                0
FLAG_EMAIL                0
OCCUPATION_TYPE        1436
CNT_FAM_MEMBERS           0
dtype: int64
Remaining missing values in occupation: 0


## 4. Feature Engineering: Demographics

In [4]:
# 4.1 Age calculation from DAYS_BIRTH
app_df['Age_Years'] = -app_df['DAYS_BIRTH'] / 365.25

# 4.2 Employment Duration calculation from DAYS_EMPLOYED
app_df['Employed_Years'] = app_df['DAYS_EMPLOYED'].apply(lambda x: 0.0 if x > 0 else -x / 365.25)

# 4.3 Family Dependency ratio
app_df['Family_Dependency'] = app_df['CNT_CHILDREN'] / (app_df['CNT_FAM_MEMBERS'] + 1e-5)

# 4.4 Income Category Binning
app_df['Income_Category'] = pd.qcut(app_df['AMT_INCOME_TOTAL'], q=4, labels=['Low', 'Medium', 'High', 'Very High']).astype(str)

# Drop raw days and mobility columns
app_df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL'], inplace=True)

print(app_df[['Age_Years', 'Employed_Years', 'Family_Dependency', 'Income_Category']].head())

   Age_Years  Employed_Years  Family_Dependency Income_Category
0  49.796030       27.687885           0.000000            High
1  53.675565       17.215606           0.499998          Medium
2  58.839151       23.496235           0.000000          Medium
3  25.522245        1.054073           0.000000          Medium
4  21.004791        2.527036           0.000000       Very High


## 5. Feature Engineering: Credit Logs Aggregation & Risk Scoring
Map statuses: 'X' and 'C' as `0`, '0' as `0`, and values '1'-'5' as risk magnitudes. Construct the target label `y` based on delinquency: `1` if the applicant ever had a delay >= 1 (STATUS is '1', '2', '3', '4', '5') else `0`.

In [5]:
status_risk_map = {'X': 0, 'C': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
credit_df['Risk_Score_Mapped'] = credit_df['STATUS'].map(status_risk_map)

# Group by ID and aggregate credit account details
credit_summary = credit_df.groupby('ID').agg(
    Credit_Window=('MONTHS_BALANCE', lambda x: x.max() - x.min()),
    Open_Month=('MONTHS_BALANCE', 'min'),
    End_Month=('MONTHS_BALANCE', 'max'),
    Risk_Score=('Risk_Score_Mapped', 'mean'), # Average risk status
    Max_Delay=('Risk_Score_Mapped', 'max')
).reset_index()

# Create target variable y: 1 if Max_Delay >= 1 (delinquency occurred) else 0 (repaid on time)
credit_summary['y'] = (credit_summary['Max_Delay'] >= 1).astype(int)
credit_summary.drop(columns=['Max_Delay'], inplace=True)

print("Credit Account Aggregations Head:")
print(credit_summary.head())
print(f"Target distribution:\n{credit_summary['y'].value_counts(normalize=True)}")

Credit Account Aggregations Head:
        ID  Credit_Window  Open_Month  End_Month  Risk_Score  y
0  5000100             15         -15          0    0.125000  1
1  5000101             16         -16          0    0.117647  1
2  5000102             32         -32          0    0.030303  1
3  5000103             24         -24          0    0.000000  0
4  5000104             21         -21          0    0.090909  1
Target distribution:
y
1    0.8232
0    0.1768
Name: proportion, dtype: float64


## 6. Merge Datasets

In [6]:
merged_df = pd.merge(app_df, credit_summary, on='ID', how='inner')
print(f"Merged Dataset Shape: {merged_df.shape}")

# Drop ID column as it is not a predictor feature
merged_df.drop(columns=['ID'], inplace=True)
print(merged_df.head(2).T)

Merged Dataset Shape: (5000, 24)
                                                 0  \
CODE_GENDER                                      F   
FLAG_OWN_CAR                                     N   
FLAG_OWN_REALTY                                  Y   
CNT_CHILDREN                                     0   
AMT_INCOME_TOTAL                         174309.78   
NAME_INCOME_TYPE              Commercial associate   
NAME_EDUCATION_TYPE  Secondary / secondary special   
NAME_FAMILY_STATUS                       Separated   
NAME_HOUSING_TYPE                House / apartment   
FLAG_WORK_PHONE                                  1   
FLAG_PHONE                                       0   
FLAG_EMAIL                                       0   
OCCUPATION_TYPE                            Unknown   
CNT_FAM_MEMBERS                                2.0   
Age_Years                                 49.79603   
Employed_Years                           27.687885   
Family_Dependency                              0.

## 7. Categorical Encoding & Save Encoders

In [7]:
categorical_cols = [
    'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'Income_Category'
]

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    merged_df[col] = le.fit_transform(merged_df[col])
    encoders[col] = le
    print(f"Encoded {col}: Classes {le.classes_}")

# Save encoder dictionary
joblib.dump(encoders, 'encoder.pkl')
print("Encoders saved successfully.")

Encoded CODE_GENDER: Classes ['F' 'M']
Encoded FLAG_OWN_CAR: Classes ['N' 'Y']
Encoded FLAG_OWN_REALTY: Classes ['N' 'Y']
Encoded NAME_INCOME_TYPE: Classes ['Commercial associate' 'Pensioner' 'State servant' 'Student' 'Working']
Encoded NAME_EDUCATION_TYPE: Classes ['Academic degree' 'Higher education' 'Incomplete higher'
 'Lower secondary' 'Secondary / secondary special']
Encoded NAME_FAMILY_STATUS: Classes ['Civil marriage' 'Married' 'Separated' 'Single / not married' 'Widow']
Encoded NAME_HOUSING_TYPE: Classes ['Co-op apartment' 'House / apartment' 'Municipal apartment'
 'Office apartment' 'Rented apartment' 'With parents']
Encoded OCCUPATION_TYPE: Classes ['Accountants' 'Cleaning staff' 'Cooking staff' 'Core staff' 'Drivers'
 'HR staff' 'High skill tech staff' 'IT staff' 'Laborers'
 'Low-skill Laborers' 'Managers' 'Medicine staff' 'Private service staff'
 'Realty agents' 'Sales staff' 'Secretaries' 'Security staff' 'Unknown'
 'Waiters/barmen staff']
Encoded Income_Category: Classes

## 8. Feature Normalization & Export
To avoid data leakage, we drop credit aggregate columns (`Credit_Window`, `Open_Month`, `End_Month`, `Risk_Score`) from training predictors `X`.

In [8]:
X = merged_df.drop(columns=['y', 'Credit_Window', 'Open_Month', 'End_Month', 'Risk_Score'])
y = merged_df['y']

feature_names = X.columns.tolist()
print("Features list:", feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save Scaler
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved successfully.")

# Export processed dataframe
processed_df = pd.DataFrame(X_scaled, columns=feature_names)
processed_df['y'] = y.values

processed_df.to_csv('processed_dataset.csv', index=False)
print(f"Processed dataset saved to processed_dataset.csv. Shape: {processed_df.shape}")

Features list: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'Age_Years', 'Employed_Years', 'Family_Dependency', 'Income_Category']
Scaler saved successfully.
Processed dataset saved to processed_dataset.csv. Shape: (5000, 19)


## 9. Plot Processed Target Variable Distribution

In [9]:
plt.figure(figsize=(6, 5))
sns.countplot(x=y, palette='Set1')
plt.title('Target Variable Distribution (y)', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.xlabel('Approval Status (0=Approved, 1=Rejected)')
plt.ylabel('Applicant Count')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=300)
plt.close()
print("Target variable distribution plot saved.")

Target variable distribution plot saved.
